# Naive Bayes: adding up evidence
**Machine Learning · Week 8 (The rest of the classical toolbox) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w8d1.html

**Naive Bayes** turns Bayes' rule into a classifier: for every class, multiply how common the class is by how
likely each feature is in that class, as if the features were independent. In logs, every word in an email
simply adds its own piece of evidence.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

spam = ["win a free prize now", "claim your free gift card today", "you won cash click now",
        "free entry to win a phone", "urgent claim your cash prize", "cheap meds click here now",
        "winner you have been selected for a free cruise", "click here to claim your reward",
        "limited offer win money fast", "get cash now no fees"]
ham = ["are we still meeting for lunch today", "can you send me the notes from class",
       "the lab meeting moved to friday", "thanks for the help with the homework",
       "see you at the library at noon", "did you finish the quantum homework",
       "lunch at noon sounds good", "here are the slides from today",
       "can we move our meeting to monday", "happy birthday hope you have a great day"]
texts = spam + ham
y = np.array([1] * len(spam) + [0] * len(ham))           # 1 = spam, 0 = not spam ("ham")

vec = CountVectorizer()
X = vec.fit_transform(texts).toarray()                    # one column per word, counts per message
words = vec.get_feature_names_out()
print(f"{len(texts)} messages, {len(words)} different words")

# P(word | class) with add-one (Laplace) smoothing, so an unseen word never gives probability 0
counts = np.array([X[y == c].sum(axis=0) for c in (0, 1)])            # (2 classes, words)
p_word = (counts + 1) / (counts.sum(axis=1, keepdims=True) + len(words))
prior = np.array([np.mean(y == 0), np.mean(y == 1)])

def log_odds(message):
    """log P(spam | message) - log P(ham | message): prior plus one term per known word."""
    total = np.log(prior[1] / prior[0])
    for w in message.split():
        if w in vec.vocabulary_:
            j = vec.vocabulary_[w]
            total += np.log(p_word[1, j] / p_word[0, j])
    return total

msg = "free lunch at noon today"
z = log_odds(msg)
print(f'by hand:      "{msg}": log-odds {z:+.2f}, P(spam) = {1 / (1 + np.exp(-z)):.3f}')
nb = MultinomialNB(alpha=1.0).fit(X, y)
print(f"scikit-learn: P(spam) = {nb.predict_proba(vec.transform([msg]))[0, 1]:.3f}")

## Which words are the strongest evidence?
Each word's vote is its **log-likelihood ratio** $\log \frac{P(\text{word}\mid\text{spam})}{P(\text{word}\mid\text{ham})}$:
positive pushes towards spam, negative towards ham.

In [ ]:
import matplotlib.pyplot as plt

llr = np.log(p_word[1] / p_word[0])
order = np.argsort(llr)
pick = np.r_[order[:8], order[-8:]]
plt.figure(figsize=(7, 4.2))
plt.barh(words[pick], llr[pick], color=np.where(llr[pick] > 0, "#ff8a50", "#5577f5"))
plt.axvline(0, color="#2c2842", lw=1)
plt.xlabel("log-likelihood ratio (spam > 0, ham < 0)")
plt.title("The strongest words for each class")
plt.show()

## Numbers instead of words: Gaussian naive Bayes, LDA and QDA
For real-valued features, model each feature in each class as a bell curve (a Gaussian). Three relatives:
**Gaussian naive Bayes** (one bell curve per feature, no correlations), **LDA** (one shared, tilted Gaussian
shape for all classes, so the boundary is a straight line) and **QDA** (each class gets its own shape, so
the boundary can curve). All three are *generative*: they model how each class produces data, then use Bayes' rule.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from matplotlib.colors import LinearSegmentedColormap

rng = np.random.default_rng(1)
A = rng.multivariate_normal([0, 0], [[0.5, 0.0], [0.0, 0.5]], 200)          # a tight round class
B = rng.multivariate_normal([1.2, 0.6], [[3.0, 1.4], [1.4, 2.4]], 200)      # a wide, tilted one around it
X2 = np.vstack([A, B])
y2 = np.r_[np.zeros(200), np.ones(200)]
Xtr, Xte, ytr, yte = train_test_split(X2, y2, test_size=0.3, random_state=0)
apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])
models = {"Gaussian naive Bayes": GaussianNB(), "LDA": LinearDiscriminantAnalysis(),
          "QDA": QuadraticDiscriminantAnalysis(), "logistic regression": LogisticRegression()}
xx, yy = np.meshgrid(np.linspace(-4, 6, 220), np.linspace(-4, 5, 200))
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), sharey=True, layout="constrained")
for ax, (name, m) in zip(axes, models.items()):
    m.fit(Xtr, ytr)
    p = m.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=20, cmap=apricot_blue, alpha=0.5)
    ax.contour(xx, yy, p, levels=[0.5], colors="#2c2842", linewidths=1.5)
    ax.scatter(Xtr[:, 0], Xtr[:, 1], c=np.where(ytr == 0, "#ff8a50", "#5577f5"), s=10)
    ax.set(xlim=(-4, 6), ylim=(-4, 5))
    ax.set_title(f"{name}\ntest accuracy {m.score(Xte, yte):.2f}", fontsize=11.5)
plt.show()

## On real data: tumors
The breast-cancer dataset has 30 measurements per tumor. Naive Bayes trains in a blink and needs no tuning;
logistic regression usually wins a little once there is enough data.

In [ ]:
import time
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Xc, yc = load_breast_cancer(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0, stratify=yc)
for name, m in [("Gaussian naive Bayes", GaussianNB()),
                ("logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))]:
    t0 = time.perf_counter()
    m.fit(Xc_tr, yc_tr)
    print(f"{name:22s} test accuracy {m.score(Xc_te, yc_te):.3f}   trained in {1000 * (time.perf_counter() - t0):.1f} ms")

## Your turn: write your own messages
Write three messages of your own: one clearly spam, one clearly not, and one that mixes both kinds of words.
Print each message's P(spam) with the `nb` model, and the three words that pushed it hardest
(the known words with the largest absolute log-likelihood ratio `llr`).

In [ ]:
my_messages = ["TODO: a spam message", "TODO: a normal message", "TODO: a mixed one"]
for m in my_messages:
    p = None    # TODO: nb.predict_proba(vec.transform([m]))[0, 1]
    known = [w for w in m.lower().split() if w in vec.vocabulary_]
    top = None  # TODO: the 3 words in `known` with the largest abs(llr[vec.vocabulary_[w]])
    print(f"{m!r}: P(spam) = {p}, strongest words: {top}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
my_messages = ["claim your free cash prize now", "can you send me the lab notes", "free pizza at the lab meeting"]
for m in my_messages:
    p = nb.predict_proba(vec.transform([m]))[0, 1]
    known = [w for w in m.lower().split() if w in vec.vocabulary_]
    top = sorted(known, key=lambda w: -abs(llr[vec.vocabulary_[w]]))[:3]
    print(f"{m!r}: P(spam) = {p:.3f}, strongest words: {top}")
print("Words the model never saw (like 'pizza') add no evidence at all: they are simply skipped.")